# Validate CNN Output

Evaluate trained model predictions against CONUS404 targets. Visualize spatial
fields, error distributions, and per-variable metrics.

**Usage:** Set `CASE_STUDY` below. Requires a trained model checkpoint.

In [ ]:
import sys
from pathlib import Path

import torch
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('..').resolve()))
from cosmos_wind_cnn.utils.config import load_config, parse_variable_config
from cosmos_wind_cnn.models.unet3d import Wind3DUNET
from cosmos_wind_cnn.data.dataset import WindDataset3D
from cosmos_wind_cnn.training.metrics import (
    calculate_rmse, calculate_mae,
    calculate_wind_speed_metrics, calculate_direction_metrics,
)

%matplotlib inline

# --- Configuration ---
CASE_STUDY = '../case_studies/sf_bay'
N_SAMPLES = 5  # Number of sample predictions to visualize

case_dir = Path(CASE_STUDY)
config = load_config(case_dir / 'configs' / 'training.yaml')
input_vars, output_vars, wind_pair_indices = parse_variable_config(config)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Case study: {case_dir.name}')
print(f'Device: {device}')
print(f'Output vars: {output_vars}')

## 1. Load model and test data

In [ ]:
# Load checkpoint
ckpt_path = case_dir / 'checkpoints' / 'best_model.pth'
if not ckpt_path.exists():
    raise FileNotFoundError(f'{ckpt_path} not found. Train the model first.')

checkpoint = torch.load(ckpt_path, map_location=device)
print(f'Loaded checkpoint from epoch {checkpoint["epoch"]}')
print(f'  Val loss: {checkpoint["val_loss"]:.4f}')

model = Wind3DUNET(
    in_channels=len(input_vars),
    out_channels=len(output_vars),
    base_channels=config.get('base_channels', 32),
).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'  Parameters: {n_params:,}')

In [ ]:
# Load test dataset
data_dir = case_dir / 'data' / 'processed'

test_dataset = WindDataset3D(
    netcdf_path=str(data_dir / 'test.nc'),
    stats_path=str(data_dir / 'normalization_stats.pkl'),
    input_vars=input_vars,
    output_vars=output_vars,
    sequence_length=config['sequence_length'],
    forecast_horizon=config['forecast_horizon'],
    stride=1,
)
print(f'Test samples: {len(test_dataset)}')

## 2. Generate predictions on test set

In [ ]:
from torch.utils.data import DataLoader

test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False)

all_preds = []
all_targets = []

with torch.no_grad():
    for inputs, targets in test_loader:
        outputs = model(inputs.to(device))
        all_preds.append(outputs.cpu())
        all_targets.append(targets.cpu())

all_preds = torch.cat(all_preds)
all_targets = torch.cat(all_targets)
print(f'Predictions shape: {all_preds.shape}')
print(f'Targets shape: {all_targets.shape}')

## 3. Per-variable metrics

In [ ]:
print(f'{"Variable":<25} {"RMSE":>10} {"MAE":>10} {"Bias":>10}')
print('-' * 55)

for i, var in enumerate(output_vars):
    pred_i = all_preds[:, i]
    tgt_i = all_targets[:, i]
    rmse = torch.sqrt(torch.mean((pred_i - tgt_i) ** 2)).item()
    mae = torch.mean(torch.abs(pred_i - tgt_i)).item()
    bias = torch.mean(pred_i - tgt_i).item()
    print(f'{var:<25} {rmse:>10.4f} {mae:>10.4f} {bias:>10.4f}')

# Wind-specific metrics
if wind_pair_indices:
    u_idx, v_idx = wind_pair_indices[0]
    wind_pred = all_preds[:, [u_idx, v_idx]]
    wind_tgt = all_targets[:, [u_idx, v_idx]]
    speed_m = calculate_wind_speed_metrics(wind_pred, wind_tgt)
    dir_m = calculate_direction_metrics(wind_pred, wind_tgt)
    print(f'\nWind speed RMSE: {speed_m["speed_rmse"]:.4f}')
    print(f'Wind speed bias: {speed_m["speed_bias"]:.4f}')
    print(f'Direction MAE:   {dir_m["direction_mae"]:.2f} deg')

## 4. Sample predictions vs targets

In [ ]:
sample_indices = np.random.choice(len(all_preds), min(N_SAMPLES, len(all_preds)), replace=False)

for var_i, var_name in enumerate(output_vars):
    for idx in sample_indices:
        pred = all_preds[idx, var_i].numpy()
        tgt = all_targets[idx, var_i].numpy()
        diff = pred - tgt

        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        vmin = min(pred.min(), tgt.min())
        vmax = max(pred.max(), tgt.max())

        im0 = axes[0].imshow(tgt, cmap='RdBu_r', vmin=vmin, vmax=vmax, origin='lower')
        axes[0].set_title('Target (CONUS404)')
        plt.colorbar(im0, ax=axes[0])

        im1 = axes[1].imshow(pred, cmap='RdBu_r', vmin=vmin, vmax=vmax, origin='lower')
        axes[1].set_title('Predicted')
        plt.colorbar(im1, ax=axes[1])

        dmax = max(abs(diff.min()), abs(diff.max()))
        im2 = axes[2].imshow(diff, cmap='RdBu_r', vmin=-dmax, vmax=dmax, origin='lower')
        axes[2].set_title('Error (Pred - Target)')
        plt.colorbar(im2, ax=axes[2])

        fig.suptitle(f'{var_name} - Sample {idx}', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

## 5. Error distributions

In [ ]:
n_vars = len(output_vars)
fig, axes = plt.subplots(1, n_vars, figsize=(5 * n_vars, 4))
if n_vars == 1:
    axes = [axes]

for i, var_name in enumerate(output_vars):
    error = (all_preds[:, i] - all_targets[:, i]).flatten().numpy()
    axes[i].hist(error, bins=60, alpha=0.7, edgecolor='black', color='steelblue')
    axes[i].axvline(0, color='red', linestyle='--', linewidth=1.5)
    axes[i].set_title(f'{var_name}', fontweight='bold')
    axes[i].set_xlabel('Prediction Error')
    axes[i].set_ylabel('Count')
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Error Distributions', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Scatter plots (predicted vs target)

In [ ]:
fig, axes = plt.subplots(1, n_vars, figsize=(5 * n_vars, 5))
if n_vars == 1:
    axes = [axes]

for i, var_name in enumerate(output_vars):
    tgt_flat = all_targets[:, i].flatten().numpy()
    pred_flat = all_preds[:, i].flatten().numpy()

    # Subsample for plotting
    n_pts = min(100_000, len(tgt_flat))
    idx = np.random.choice(len(tgt_flat), n_pts, replace=False)

    axes[i].hexbin(tgt_flat[idx], pred_flat[idx], gridsize=50, cmap='Blues', mincnt=1)

    lo = min(tgt_flat[idx].min(), pred_flat[idx].min())
    hi = max(tgt_flat[idx].max(), pred_flat[idx].max())
    axes[i].plot([lo, hi], [lo, hi], 'r--', linewidth=1.5, label='1:1')

    r2 = 1 - np.sum((pred_flat[idx] - tgt_flat[idx]) ** 2) / np.sum((tgt_flat[idx] - tgt_flat[idx].mean()) ** 2)
    axes[i].text(0.05, 0.95, f'R2={r2:.3f}', transform=axes[i].transAxes,
                 fontsize=10, va='top',
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    axes[i].set_title(f'{var_name}', fontweight='bold')
    axes[i].set_xlabel('Target')
    axes[i].set_ylabel('Predicted')
    axes[i].legend()
    axes[i].set_aspect('equal', adjustable='box')
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Predicted vs Target', fontsize=14)
plt.tight_layout()
plt.show()